# pivotai — Baseline Comparison (Phase 4 Extension)

Runs the **untuned `llama3.1:8b`** base model against the same 92 golden test cases used to evaluate `pivotai-ft`, `pivotai-distill`, and `pivotai-curriculum`.

**Purpose:** Establish the pre-training performance floor. The gap between baseline and trained models quantifies what fine-tuning actually achieved.

**Runtime:** ~3–4 hours on Colab T4 (CPU-mode Ollama, 92 × ~100s per inference).

## Steps
1. Mount Drive and clone the repo
2. Install dependencies + Ollama
3. Pull `llama3.1:8b` model
4. Generate responses for all 92 golden records
5. Score responses (no LLM judge — local metrics only)
6. Print 4-model comparison table
7. Download results to Drive

In [ ]:
# ── Cell 1: Mount Drive + clone repo ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/pivotai'
REPO_DIR  = '/content/travel_project'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Clone (or pull) the repo
if not os.path.exists(REPO_DIR):
    # Replace with your actual GitHub repo URL after pushing
    !git clone https://github.com/ishreyadev/pivotai.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 2: Install Python dependencies ───────────────────────────────────────
!pip install -q httpx tqdm sentence-transformers rouge-score bert-score python-dotenv pydantic

In [ ]:
# ── Cell 3: Install Ollama + pull llama3.1:8b ─────────────────────────────────
# Install Ollama (Linux/Colab)
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # wait for server to be ready
print('Ollama server started (PID:', proc.pid, ')')

# Pull the base model (~4.7 GB — takes 5-10 min on Colab)
print('Pulling llama3.1:8b ...')
!ollama pull llama3.1:8b
print('Done.')

In [ ]:
# ── Cell 4: Copy golden set from Drive (if not already in data/evals/) ────────
import shutil, os

EVALS_DIR   = f'{REPO_DIR}/data/evals'
GOLDEN_PATH = f'{EVALS_DIR}/golden_set.jsonl'

# If you backed up data/evals to Drive, copy it back
DRIVE_EVALS = f'{DRIVE_DIR}/data/evals'
if not os.path.exists(GOLDEN_PATH) and os.path.exists(f'{DRIVE_EVALS}/golden_set.jsonl'):
    os.makedirs(EVALS_DIR, exist_ok=True)
    shutil.copy(f'{DRIVE_EVALS}/golden_set.jsonl', GOLDEN_PATH)
    print('Copied golden_set.jsonl from Drive')

if os.path.exists(GOLDEN_PATH):
    with open(GOLDEN_PATH) as f:
        n = sum(1 for _ in f)
    print(f'Golden set: {n} records')
else:
    print('ERROR: golden_set.jsonl not found. Upload it to data/evals/ or copy from Drive.')

In [ ]:
# ── Cell 5: Generate baseline responses (92 records × ~100s ≈ 2.5 hrs) ───────
# Uses the existing generate_responses.py pipeline — no code changes needed.
# --no-judge skips the DeepSeek judge to avoid API costs; re-run with judge later if needed.

!python phase4_evals/generate_responses.py --model llama3.1:8b

In [ ]:
# ── Cell 6: Score baseline responses ──────────────────────────────────────────
# --no-judge: skip LLM judge (saves API quota; adds reasoning_coherence/grounding later)
!python phase4_evals/score_responses.py --no-judge

In [ ]:
# ── Cell 7: 4-model comparison table (--no-h2h to skip judge calls) ──────────
!python phase4_evals/compare.py --no-h2h

In [ ]:
# ── Cell 8: Quick summary — trained vs baseline gap ───────────────────────────
import json, glob, sys
sys.path.insert(0, REPO_DIR)

from config import EVALS_DIR as EVALS_PATH
EVALS_PATH = str(EVALS_PATH)

# Load latest summary
summaries = sorted(glob.glob(f'{EVALS_PATH}/summary_*.json'))
if not summaries:
    print('No summary file found — run compare.py first')
else:
    with open(summaries[-1]) as f:
        summary = json.load(f)

    models = summary.get('models', {})
    baseline = models.get('llama3.1:8b', {})
    ft       = models.get('pivotai-ft', {})

    print('=== Baseline vs pivotai-ft gap ===')
    for metric in ['json_valid', 'savings_valid', 'budget_compliance',
                   'schema_compliance', 'bertscore_f1', 'rouge_l',
                   'reasoning_coherence', 'grounding_accuracy']:
        bval = baseline.get(metric)
        fval = ft.get(metric)
        if bval is not None and fval is not None:
            delta = fval - bval
            sign  = '+' if delta >= 0 else ''
            print(f'  {metric:<28} baseline={bval:.3f}  ft={fval:.3f}  ({sign}{delta:+.3f})')
        else:
            print(f'  {metric:<28} baseline={bval}  ft={fval}')

In [ ]:
# ── Cell 9: Save results to Drive ─────────────────────────────────────────────
import shutil, glob, os

os.makedirs(f'{DRIVE_DIR}/data/evals', exist_ok=True)

# Copy all new eval files (responses + scored results + summary)
for pattern in ['responses_*.jsonl', 'eval_results_*.jsonl', 'summary_*.json']:
    for f in glob.glob(f'{EVALS_PATH}/{pattern}'):
        dest = f'{DRIVE_DIR}/data/evals/{os.path.basename(f)}'
        shutil.copy(f, dest)
        print(f'Saved: {os.path.basename(f)}')

print('\nAll results saved to Drive.')
print('Next: copy updated data/evals/ back to your local machine, then run:')
print('  python phase4_evals/compare.py')
print('  python phase4_evals/notebooks/results_analysis.ipynb  (local Jupyter)')